# OASIS Colab Training

This notebook runs the OASIS MONAI training pipeline in Google Colab.

It keeps the same backend structure we use locally:
- OASIS stays separate from Kaggle
- decision-support only
- outputs are saved back into the repo under `outputs/runs/oasis/`


## 1. Mount Drive And Set Paths

Edit `DRIVE_REPO_ROOT` and `DRIVE_OASIS_ROOT` to match your Google Drive layout.


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

DRIVE_REPO_ROOT = '/content/drive/MyDrive/YOUR_PATH/alz_backend'
DRIVE_OASIS_ROOT = '/content/drive/MyDrive/YOUR_PATH/OASIS'
RUN_NAME = 'oasis_colab_v1'

repo_root = Path(DRIVE_REPO_ROOT)
repo_script = repo_root / 'scripts' / 'train_oasis_colab.py'
requirements_path = repo_root / 'requirements-colab.txt'

if not repo_root.exists():
    raise FileNotFoundError(
        f'{repo_root} does not exist. Point DRIVE_REPO_ROOT at the folder containing requirements-colab.txt.'
    )
if not repo_script.exists() or not requirements_path.exists():
    raise FileNotFoundError(
        f'{repo_root} is not an alz_backend checkout. Expected {repo_script} and {requirements_path}.'
    )

print('repo_root:', repo_root)
print('oasis_root:', DRIVE_OASIS_ROOT)


## 2. Check GPU Runtime


In [ ]:
!nvidia-smi

import torch
print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())
print('device_count:', torch.cuda.device_count())
print('device_name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')


## 3. Install Backend Dependencies

`requirements-colab.txt` intentionally leaves PyTorch to the CUDA build already present in Colab.


In [ ]:
%cd {repo_root}
!pip install -r requirements-colab.txt


## 4. Run OASIS Training And Evaluation

Default Colab config:
- `device=cuda`
- `mixed_precision=true`
- `batch_size=4`
- `image_size=64 64 64`

If Colab RAM is tight, lower `--batch-size` to `2`.


In [ ]:
%cd {repo_root}
!python scripts/train_oasis_colab.py \
  --oasis-source-dir "{DRIVE_OASIS_ROOT}" \
  --run-name "{RUN_NAME}" \
  --evaluate-splits val test


## 5. Inspect The Run


In [ ]:
from pathlib import Path
import json

run_root = Path(DRIVE_REPO_ROOT) / 'outputs' / 'runs' / 'oasis' / RUN_NAME
print('run_root:', run_root)
print('exists:', run_root.exists())
print('files:', sorted(p.name for p in run_root.iterdir()))

metrics_path = run_root / 'evaluation' / 'post_train_test_best_model' / 'metrics.json'
if metrics_path.exists():
    print(json.dumps(json.loads(metrics_path.read_text()), indent=2))
else:
    print('metrics not found yet')


## 6. Calibrate The Threshold

This picks a threshold from validation predictions and applies it to the test split.


In [ ]:
%cd {repo_root}
!python scripts/calibrate_oasis_threshold.py \
  --validation-predictions "outputs/runs/oasis/{RUN_NAME}/evaluation/post_train_val_best_model/predictions.csv" \
  --test-predictions "outputs/runs/oasis/{RUN_NAME}/evaluation/post_train_test_best_model/predictions.csv" \
  --output-dir "outputs/runs/oasis/{RUN_NAME}/calibration/threshold_f1" \
  --selection-metric f1


## 7. Promote The Run As The Current Baseline

Only do this if the Colab run is better than the current promoted baseline.


In [ ]:
%cd {repo_root}
!python scripts/promote_oasis_checkpoint.py \
  --run-name "{RUN_NAME}" \
  --checkpoint-path "outputs/runs/oasis/{RUN_NAME}/checkpoints/best_model.pt" \
  --val-metrics-path "outputs/runs/oasis/{RUN_NAME}/evaluation/post_train_val_best_model/metrics.json" \
  --test-metrics-path "outputs/runs/oasis/{RUN_NAME}/evaluation/post_train_test_best_model/metrics.json" \
  --threshold-calibration-path "outputs/runs/oasis/{RUN_NAME}/calibration/threshold_f1/threshold_calibration.json" \
  --image-size 64 64 64


## 8. Load The Promoted Model Registry


In [ ]:
from pathlib import Path
import json

registry_path = Path(DRIVE_REPO_ROOT) / 'outputs' / 'model_registry' / 'oasis_current_baseline.json'
print(json.dumps(json.loads(registry_path.read_text()), indent=2))
